In [0]:
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text("source_system","")
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_table_name","")
dbutils.widgets.text("source_file_format","")


source_system = dbutils.widgets.get("source_system")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_table_name = dbutils.widgets.get("source_table_name")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
source_system_path = f"/Volumes/caresyncsydatabricks/landing/landing/{source_system}/"
landing_timestamp_path = f"/Volumes/caresyncsydatabricks/landing/landing/{source_system}/{source_table_name}/{landing_timestamp}"
source_table_name_path = f"/Volumes/caresyncsydatabricks/landing/landing/{source_system}/{source_table_name}/"

try:
    if all(dbutils.fs.ls(path) for path in [source_system_path, landing_timestamp_path, source_table_name_path]):
        print("All paths found")
        

except:
    print("Path not found")
    dbutils.notebook.exit("Some error found in path")



In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv")\
                      .option("header", "true")\
                      .option("inferSchema", "true")\
                      .load(f"/Volumes/caresyncsydatabricks/landing/landing/{source_system}/{source_table_name}/{landing_timestamp}/")
else:
    landing_df = spark.read.format("parquet")\
                      .load(f"/Volumes/caresyncsydatabricks/landing/landing/{source_system}/{source_table_name}/{landing_timestamp}/")

In [0]:
updated_df = landing_df.withColumn("landing_timestamp",lit(landing_timestamp))\
                         .withColumn("updated_timestamp",current_timestamp())

In [0]:
updated_df.write.mode("append")\
            .saveAsTable(f"caresyncsydatabricks.bronze.{source_table_name}")